In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [ ]:
sales = pd.read_csv(r"data/SalesData.csv")

In [ ]:
sales

### Data Analysis

In [ ]:
# To check the dimensions

sales.shape

In [ ]:
# To check column names with their datatypes

sales.info()

In [ ]:
# To check for null/missing values

sales.isna().sum()

In [ ]:
# To check for duplicate records

sales.duplicated().sum()

In [ ]:
# To generate summary statistics

sales.describe().T

### Data Cleaning

In [ ]:
# To check unique categories

print("Regions:", sales["Region"].unique())
print("\nTiers:", sales["Tier"].unique())
print("\nMonths:", sales["Month"].unique())
print("\nDivisions:", sales["Division"].unique())

In [ ]:
# To get month no of every month for chronological order of data.

month_order = {"Jan": 1, "Feb": 2, "Mar": 3,"Apr": 4, "May": 5, "Jun": 6,"Jul": 7, "Aug": 8, "Sep": 9,"Oct": 10, "Nov": 11, "Dec": 12}

sales.insert(sales.columns.get_loc("Month")+1,"MonthNo",sales["Month"].map(month_order))


In [ ]:
# 2nd approach

sales["Month"] = pd.Categorical(sales["Month"],categories=month_order,ordered=True)

In [ ]:
sales.head()

### Exploratory Data Analysis (EDA)

In [ ]:
# To calculate overall sales and unit totals for 2015 and 2016

sales_2015 = sales["Sales2015"].sum()
sales_2016 = sales["Sales2016"].sum()

units_2015 = sales["Units2015"].sum()
units_2016 = sales["Units2016"].sum()

print("Total Sales 2015:", round(sales_2015,3))
print("Total Sales 2016:", round(sales_2016,3))

print("Total Units 2015:", round(units_2015,3))
print("Total Units 2016:", round(units_2016,3))

In [ ]:
# To calculate Year-over-Year(YoY) Growth

sales_growth = ((sales_2016 - sales_2015) / sales_2015) * 100

print("Sales Growth (YoY):", round(sales_growth, 2), "%")

### Answering Business Problems

##### 1. Which regions drove growth in 2016 — and which dragged performance down?

In [ ]:
# Calculating sales by region

regional_sales = sales.groupby("Region")[["Sales2015","Sales2016"]].sum()

regional_sales.style.format("{:.3f}")


In [ ]:
# To calculate sales change and growth percentage

regional_sales["Sales_Change"] = (regional_sales["Sales2016"] - regional_sales["Sales2015"])

regional_sales["Growth_Percentage"] = (regional_sales["Sales_Change"] / regional_sales["Sales2015"]) * 100

regional_sales.style.format("{:,.3f}")

In [ ]:
# Visual - compares the percentage growth of each region, making it easier to identify which region grew fastest relative to its 2015 sales.

regional_sales["Growth_Percentage"].plot(kind="bar")

plt.title("Regional Sales Growth Percentage in 2016")
plt.xlabel("Region")
plt.ylabel("Sales Growth (%)")


##### 2. Are all customer tiers contributing proportionally to revenue, or is the business overly dependent on a single segment?

In [ ]:
# Calculating sales by tier

tier_sales = sales.groupby("Tier")[["Sales2015","Sales2016"]].sum()

tier_sales.style.format("{:.3f}")

In [ ]:
# Calculating each tier's contribution 

tier_contribution = (tier_sales.div(tier_sales.sum(axis=0), axis=1) * 100) # axis=0 means sum down the rows,axis=1 works across columns

tier_contribution.round(3)

In [ ]:
# Visual of contribution

tier_contribution.plot(kind="bar")
plt.title("Customer Tier Contribution to Revenue: 2015 vs 2016")
plt.xlabel("Customer Tier")
plt.ylabel("Revenue Contribution (%)")

##### 3. Are there specific states where the company's position has weakened, signalling competitive pressure or poor execution?

In [ ]:
# Calculating sales by state

state_sales = sales.groupby("State")[["Sales2015", "Sales2016"]].sum()

state_sales.style.format("{:.3f}")

In [ ]:
# Calculating sales change and growth percentage

state_sales["Sales_Change"] = (state_sales["Sales2016"] - state_sales["Sales2015"])

state_sales["Growth_Percentage"] = (state_sales["Sales_Change"] / state_sales["Sales2015"]) * 100

state_sales.round(3)

In [ ]:
# To identify where sales declined

declining_sales = state_sales[state_sales["Sales_Change"] < 0]

declining_sales.sort_values("Sales_Change")

In [ ]:
# Visual

declining_sales[["Sales2015","Sales2016"]].plot(kind="bar")

plt.title("State-wise Sales Comparison: 2015 vs 2016")
plt.xlabel("State")
plt.ylabel("Total Sales")


##### 4. Has sales momentum been consistent through the year, or are there seasonal patterns that the planning team is not currently accounting for?

In [ ]:
# Calculating monhtly sales

monthly_sales = sales.groupby("MonthNo")[["Sales2015", "Sales2016"]].sum()

monthly_sales.style.format("{:.3f}")

In [ ]:
# Visualize

monthly_sales.plot(kind="line")

plt.title("Monthly Sales Trend: 2015 vs 2016")
plt.xlabel("Month Number")
plt.ylabel("Total Sales")
plt.grid(True)

In [ ]:
# Identifyfing the highest and lowest performing months

print("Highest Sales Month in 2015:",monthly_sales["Sales2015"].idxmax())

print("\nLowest Sales Month in 2015:",monthly_sales["Sales2015"].idxmin())

print("\nHighest Sales Month in 2016:",monthly_sales["Sales2016"].idxmax())

print("\nLowest Sales Month in 2016:",monthly_sales["Sales2016"].idxmin())

In [ ]:
# To get month name

import calendar

print("Highest Sales Month in 2015:",calendar.month_name[monthly_sales["Sales2015"].idxmax()])

print("\nLowest Sales Month in 2015:",calendar.month_name[monthly_sales["Sales2015"].idxmin()])

print("\nHighest Sales Month in 2016:",calendar.month_name[monthly_sales["Sales2016"].idxmax()])

print("\nLowest Sales Month in 2016:",calendar.month_name[monthly_sales["Sales2016"].idxmin()])

##### 5. In divisions where unit volumes declined, is the drop concentrated in certain product lines or accounts — and does this align with the High-value account strategy?

In [ ]:
# Calculating unit sales by division

division_units = sales.groupby("Division")[["Units2015", "Units2016"]].sum()

division_units

In [ ]:
# Calculating change in unit sales

division_units["Units_Change"] = (division_units["Units2016"] - division_units["Units2015"])

division_units["Growth_Percentage"] = (division_units["Units_Change"] / division_units["Units2015"]) * 100

division_units.round(2)

In [ ]:
# Divisons with declining sales

declining_divisions = division_units[division_units["Units_Change"] < 0]

declining_divisions.sort_values("Units_Change")

In [ ]:
# Checking for individual acccounts because overall divisions grew

account_units = sales.groupby(["Division", "AccountName", "Tier"])[["Units2015", "Units2016"]].sum()

account_units["Units_Change"] = (account_units["Units2016"] - account_units["Units2015"])

declining_accounts = account_units[account_units["Units_Change"] < 0].sort_values("Units_Change")

declining_accounts

In [ ]:
# Identifying the individual High-tier accounts that declined

high_tier_accounts = sales[sales["Tier"] == "High"].groupby(["Division", "AccountName"])[["Units2015", "Units2016"]].sum()

high_tier_accounts["Units_Change"] = (high_tier_accounts["Units2016"]- high_tier_accounts["Units2015"])

high_tier_accounts[high_tier_accounts["Units_Change"] < 0].sort_values("Units_Change")

### <center> Visual Intellligence Report </center>

#### 1.	Regional Sales Comparison

In [ ]:
for region in regional_sales.index:
    change = regional_sales.loc[region, "Sales_Change"]
    growth = regional_sales.loc[region, "Growth_Percentage"]

    if change > 0:
        print(region, "drove growth:",round(growth, 3),"% increase")
    elif change < 0:
        print(region, "dragged performance down:",round(abs(growth), 3),"% decrease")
    else:
        print(region, "showed no change in sales")

In [ ]:
regional_sales[["Sales2015", "Sales2016"]].plot(kind="bar")

plt.title("Regional Sales Comparison: 2015 vs 2016")
plt.ylabel("Total Sales")
plt.legend(["2015", "2016"])

West recorded the highest year-on-year growth at approximately 34.77%, followed by East at 33.16% and Central at 24.03%. West and East were the strongest-growing regions, while Central showed comparatively slower growth and may require further investigation to identify opportunities for improvement.

#### 2. Regional Sales Composition (2016)

In [ ]:
regional_sales_2016 = sales.groupby("Region")[["Sales2016"]].sum()

regional_sales_2016.style.format("{:.3f}")

In [ ]:
regional_sales_2016.plot(kind = "pie",y="Sales2016",wedgeprops={"width": 0.4},autopct="%1.1f%%")

plt.title("Regional Sales Composition (2016)")

The East region contributed the largest share of NovaPharma's total sales in 2016, generating approximately 12.67 million, or 42.70% of national sales, Central contributed approximately 9.79 million (33.00%), while West accounted for 7.21 million (24.30%). This indicates that East was the company's largest regional revenue contributor in 2016, while West had the smallest share and may warrant further investigation to identify opportunities for growth.

#### 3. Region × Tier Performance

In [ ]:
region_tier_sales = sales.groupby(["Region","Tier"])[["Sales2015","Sales2016"]].sum()  
# the loop goes through each region name stored in the index, one by one.

region_tier_sales.style.format("{:.3f}")

In [ ]:
region_tier_sales.unstack("Tier").plot(kind="bar",stacked=True)

plt.title("Region × Tier Sales Performance (2015 vs 2016)")
plt.xlabel("Region and Tier")
plt.ylabel("Total Sales")

plt.legend(title="Customer Tier",loc ="upper left",bbox_to_anchor=(1, 1))

Sales increased from 2015 to 2016 across the customer tiers in the Central, East, and West regions, indicating overall growth across the regional customer segments. High-tier accounts contributed the largest share of sales, reflecting their importance to NovaPharma's revenue. We should continue prioritising high-value accounts while investigating opportunities to strengthen sales from Medium- and Low-tier customers across the regions..

#### 4. East Region State-Level Analysis

In [ ]:
east_sales = sales[sales.Region == "East"]

east_state_sales = east_sales.groupby("State")[["Sales2015","Sales2016"]].sum()

east_state_sales["Sales_Change"] = (east_state_sales["Sales2016"] - east_state_sales["Sales2015"])

east_state_sales["Growth_Percentage"] = (east_state_sales["Sales_Change"] / east_state_sales.Sales2015) * 100

east_state_sales.round(3)

In [ ]:
# To calculate declining sales state

declining_state = east_state_sales[east_state_sales["Sales_Change"] < 0]

declining_state

In [ ]:
declining_state["Sales_Change"].plot(kind ="barh", title="East Region: States with Sales Decline (2015–2016)",xlabel="Change in Sales",
            ylabel="State",color="indianred",width=0.2)


**New York (NY) was the only state in the East region to experience a decline in sales from 2015 to 2016**. Sales decreased by approximately 22,279.27, representing a 1.29% decline. We should investigate the factors behind this drop and take targeted measures to improve sales performance in New York.

#### 5. High-Tier Division Unit Analysis

In [ ]:
high_tier_division = sales[sales["Tier"] == "High"].groupby("Division")[["Units2015", "Units2016"]].sum()

high_tier_division

In [ ]:
# Change in High-tier unit sales

high_tier_division["Units_Change"] = (high_tier_division["Units2016"]- high_tier_division["Units2015"])

high_tier_division["Growth_Percentage"] = (high_tier_division["Units_Change"] / high_tier_division["Units2015"]) * 100

high_tier_division.round(2)

In [ ]:
# Identifying divisons where High-tier sales declined

high_tier_declining = high_tier_division[high_tier_division["Units_Change"] < 0].sort_values("Units_Change")

high_tier_declining

In [ ]:
sns.heatmap(
    high_tier_division[["Units2015", "Units2016"]],
    annot=True,
    fmt=".0f",
    cmap="YlGnBu"  # Uses a color gradient from yellow to green to blue.
)

plt.title("High-Tier Division Unit Analysis (2015 vs 2016)")
plt.xlabel("Total Units")
plt.ylabel("Division")

The heatmap compares unit volumes across all High-tier divisions in 2015 and 2016, highlighting differences in sales volume and year-on-year performance. All divisions recorded an increase in units sold in 2016, indicating consistent growth across NovaPharma’s High-tier customer segment. Raceway showed the highest growth rate at approximately **66.54%**, followed by Gulf Stream at **62.70%**. 

#### 6. Quarterly Sales Trend

In [ ]:
# 1st approach

quarter_map = {"Jan": "Q1", "Feb": "Q1", "Mar": "Q1","Apr": "Q2", "May": "Q2", "Jun": "Q2","Jul": "Q3", "Aug": "Q3", "Sep": "Q3","Oct": "Q4", 
               "Nov": "Q4", "Dec": "Q4"}


sales.insert(sales.columns.get_loc("MonthNo") + 1,"Quarter",sales["Month"].map(quarter_map))

In [ ]:
'''

# 2nd approach

conditions = [
    sales["Month"].isin(["Jan", "Feb", "Mar"]),
    sales["Month"].isin(["Apr", "May", "Jun"]),
    sales["Month"].isin(["Jul", "Aug", "Sep"]),
    sales["Month"].isin(["Oct", "Nov", "Dec"])
]

sales.insert(sales.columns.get_loc("MonthNo") + 1,"Quarter",np.select(conditions, ["Q1", "Q2", "Q3", "Q4"], default=" "))

'''

In [ ]:
'''

# 3rd approach

quarter_map = {"Q1": ["Jan", "Feb", "Mar"],"Q2": ["Apr", "May", "Jun"],"Q3": ["Jul", "Aug", "Sep"],"Q4": ["Oct", "Nov", "Dec"]}

sales.insert(sales.columns.get_loc("MonthNo") + 1,"Quarter","")

for quarter, months in quarter_map.items():
    for month in months:
        sales.loc[sales["Month"] == month, "Quarter"] = quarter

'''

In [ ]:
'''

# 4th approach- Creating directly using month no

sales.insert(sales.columns.get_loc("MonthNo") + 1,"Quarter","Q" + ((sales["MonthNo"] - 1) // 3 + 1).astype(str))

'''

In [ ]:
sales.columns

In [ ]:
quarterly_sales = sales.groupby("Quarter")[["Sales2015", "Sales2016"]].sum()

quarterly_sales.style.format("{:.3f}")

In [ ]:
quarterly_sales.plot(kind="bar")

plt.title("Quarterly Sales Comparison: 2015 vs 2016")
plt.xlabel("Quarter")
plt.ylabel("Total Sales")


It highlights changes in performance across the four quarters.This analysis helps identify stronger and weaker quarters, enabling to refine the sales strategies and plan resources more effectively.Q3 recorded the highest sales in both years, increasing from approximately 6.16 million in 2015 to 7.86 million in 2016, highlighting a strong mid-year sales period. 

#### 7. Quarterly Tier Composition

In [ ]:
quarterly_tier_sales = sales.groupby(["Quarter","Tier"])[["Sales2016"]].sum().unstack()  # unstack() --> moves an index level into columns.

quarterly_tier_sales.style.format("{:.3f}")

In [ ]:
# Converting sales total into percentage

quarterly_tier_pct = quarterly_tier_sales.div(quarterly_tier_sales.sum(axis=1), axis=0) * 100

quarterly_tier_pct

In [ ]:
quarterly_tier_pct.plot(
    kind="bar",
    stacked=True,
    figsize=(8,6),
    title="Quarterly Tier Composition (2016)",
    xlabel="Quarter",
    ylabel="Sales Share(%)"
)

plt.legend(title="Customer Tier",loc ="upper left",bbox_to_anchor=(1, 1))

The stacked bar chart compares the quarterly sales composition of the four customer tiers in 2016, showing how each tier contributed to total sales in Q1, Q2, Q3, and Q4. The relative sizes of the stacked segments reveal differences in tier contributions across quarters and highlight which customer tiers played a larger role in sales performance.The High-tier segment’s share increased from approximately **55.27% in Q1 to 62.63% in Q4**, indicating its growing contribution to the overall sales mix. However, Q3 shows a negative contribution from the Out tier, with sales of **−3,248.53**, which may reflect adjustments or returns and should be investigated before drawing conclusions about that tier’s performance.
We should focus on retaining high-value customers while examining the Q3 anomaly to ensure accurate sales reporting and support better quarterly planning.